## 分别测试数据库连接池是否使用的情况

In [1]:
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker
from sqlalchemy import text
from sqlalchemy.pool import NullPool
from app.core.config import settings

DATABASE_URL = settings.db.url

# 不使用连接池
no_pool_engine = create_async_engine(
    DATABASE_URL,
    poolclass=NullPool,
)

# 使用连接池
pool_engine = create_async_engine(
    DATABASE_URL,           # 连接池中长期保留的连接数
    pool_size=5,            # 连接池中长期保留的连接数
    max_overflow=10,        # 连接池满时，允许临时创建的额外连接数
    pool_timeout=30,        # 获取不到连接时最多等待30秒
    pool_recycle=1800,      # 连接使用超过1800秒后重新创建
    pool_pre_ping=True,     # 取出连接前检查连接是否有效
)


async def print_connection_ids(engine, title: str) -> None:
    print(f"\n===== {title} =====")
    # 初始化session_maker
    session_maker = async_sessionmaker(engine)
    # 循环3次，获取3次连接，分别查看连接的id
    for i in range(3):
        async with session_maker() as session:
            connection_id = await session.scalar(
                text("SELECT pg_backend_pid()")
            )

            print(
                f"第 {i + 1} 次查询，"
                f"数据库连接 ID：{connection_id}"
            )


await print_connection_ids(no_pool_engine, "不使用连接池")

await print_connection_ids(pool_engine, "使用连接池")

# 关闭连接池
await no_pool_engine.dispose()
await pool_engine.dispose()


===== 不使用连接池 =====
第 1 次查询，数据库连接 ID：9949
第 2 次查询，数据库连接 ID：9950
第 3 次查询，数据库连接 ID：9951

===== 使用连接池 =====
第 1 次查询，数据库连接 ID：9952
第 2 次查询，数据库连接 ID：9952
第 3 次查询，数据库连接 ID：9952
